# SVAGA: 10K localized repair fine-tuning — Qwen 1.5B
One epoch on 10,000 official SWE-smith-derived examples, with 1,000 development-validation examples.
This is experimental localized Python repair supervision. Inputs contain buggy functions and statically selected supporting context;
they are not complete applications. Targets reverse upstream bug patches. Only three source
tasks were replayed locally. Missing surrounding code and semantic duplicates remain limitations.

Use a fresh GPU runtime. Run cells in order. This starts a **fresh adapter**, separate from
the previous 160-example terminal-action adapter. No commands in dataset text are executed.
Model: Qwen/Qwen2.5-Coder-1.5B-Instruct; 4-bit NF4; LoRA rank 8 on q/v projections;
learning rate 2e-5; batch size 1; accumulation 8; 1,250 optimizer updates; no truncation.
Checkpoints go to Google Drive after update 1 and every 50 updates (400 examples).
Loss is a prediction metric, not proof that generated repairs pass tests.

Dataset v2 uses complete affected functions, class scaffolding and referenced module definitions, with repository-balanced selection. This still does not prove semantic completeness or correct generated repairs.


In [ ]:
import sys, subprocess, importlib.util
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))
missing = [p for p in ["transformers", "peft", "accelerate", "bitsandbytes"]
           if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
import importlib.metadata
software = {p: importlib.metadata.version(p)
            for p in ["torch", "transformers", "peft", "accelerate", "bitsandbytes"]}
print(software)
import bitsandbytes as bnb
layer = bnb.nn.Linear4bit(64, 32, bias=False, quant_type="nf4").cuda()
with torch.no_grad():
    assert layer(torch.randn(1, 64, device="cuda", dtype=torch.float16)).shape == (1, 32)
del layer
torch.cuda.empty_cache()
print("4-bit GPU check passed.")


## Upload the dataset ZIP
Select only **swesmith-context-repair-10k-v2.zip**. This is a different package from the original
long OpenThoughts conversations. The cell validates the ZIP contents and file hashes.


In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile, json, hashlib
uploaded = files.upload()
assert len(uploaded) == 1, "Upload only the prepared dataset ZIP."
data_bytes = next(iter(uploaded.values()))
dataset_dir = Path("/content/svaga-localized-10k-data")
dataset_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(data_bytes)) as archive:
    assert len(archive.namelist()) == 3
    assert set(archive.namelist()) == {"manifest.json", "train.jsonl", "validation.jsonl"}
    assert sum(x.file_size for x in archive.infolist()) < 500_000_000
    for name in archive.namelist():
        (dataset_dir / name).write_bytes(archive.read(name))
manifest = json.loads((dataset_dir / "manifest.json").read_text())
assert manifest["counts"] == {"train": 10000, "validation": 1000}
for name, expected in manifest["files_sha256"].items():
    assert hashlib.sha256((dataset_dir / name).read_bytes()).hexdigest() == expected
print("10,000 training + 1,000 validation records: hashes verified.")
print("Status:", manifest["status"])
del uploaded, data_bytes


## Persist checkpoints in Drive
Keep RUN_ROOT for this experiment. For a disconnected runtime, upload the same dataset,
set RESUME_FROM to a checkpoint folder containing COMPLETE, and rerun the setup cells.
Only load training_state.pt files created by your own notebook.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUN_ROOT = Path("/content/drive/MyDrive/SVAGA/context-qwen15b-10k-v2")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
RESUME_FROM = None  # Example: str(RUN_ROOT / "step-00050")
if RESUME_FROM is None:
    assert not list(RUN_ROOT.glob("step-*")), "Existing checkpoints: set RESUME_FROM or use a new RUN_ROOT."
else:
    assert (Path(RESUME_FROM) / "COMPLETE").exists()
(RUN_ROOT / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2))
(RUN_ROOT / "software.json").write_text(json.dumps(software, indent=2))
print("Persistent output:", RUN_ROOT)


In [ ]:
"""Notebook runtime for the experimental 10K localized repair dataset.

This file is embedded into the generated Colab notebook, not executed locally.
"""
import gc
import hashlib
import json
import math
from pathlib import Path
import random
import time
import uuid
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

MODEL_ID='Qwen/Qwen2.5-Coder-1.5B-Instruct'
REVISION='2e1fd397ee46e1388853d2af2c993145b0f1098a'
SEED=42
ACCUMULATION=8
LEARNING_RATE=2e-5


def load_dataset(folder):
    folder=Path(folder)
    manifest=json.loads((folder/'manifest.json').read_text())
    assert manifest['tokenizer']==MODEL_ID and manifest['tokenizer_revision']==REVISION
    rows={}
    for split,n in [('train',10000),('validation',1000)]:
        path=folder/f'{split}.jsonl'
        assert hashlib.sha256(path.read_bytes()).hexdigest()==manifest['files_sha256'][path.name]
        rows[split]=[json.loads(x) for x in path.open(encoding='utf-8')]
        assert len(rows[split])==n
        assert len({r['instance_id'] for r in rows[split]})==n
    for key in ['instance_id','repo','bug_patch_sha256','buggy_excerpts_sha256']:
        assert not {r[key] for r in rows['train']}&{r[key] for r in rows['validation']},key
    return manifest,rows


def prepare(rows,tokenizer):
    examples=[]
    for row in rows:
        messages=row['messages']
        full=tokenizer.apply_chat_template(messages,tokenize=True,return_dict=True,truncation=False)['input_ids']
        prompt=tokenizer.apply_chat_template(messages[:1],tokenize=True,return_dict=True,add_generation_prompt=True,truncation=False)['input_ids']
        assert full[:len(prompt)]==prompt and 0<len(prompt)<len(full)<=4096
        assert len(full)==row['tokens']['total'] and len(full)-len(prompt)==row['tokens']['target']
        examples.append({'task':row['instance_id'],'input_ids':full,'prompt_length':len(prompt)})
    return examples


def batch_for(example):
    ids=torch.tensor([example['input_ids']],dtype=torch.long,device='cuda')
    labels=ids.clone();labels[:,:example['prompt_length']]=-100
    return {'input_ids':ids,'attention_mask':torch.ones_like(ids),'labels':labels}


def evaluate(model,examples):
    was_training=model.training;model.eval();total_loss=0.;targets=0;details=[]
    try:
        with torch.inference_mode():
            for example in examples:
                batch=batch_for(example)
                count=int((batch['labels'][:,1:]!=-100).sum())
                with torch.autocast('cuda',dtype=torch.float16):
                    output=model(**batch,use_cache=False)
                loss=float(output.loss.float())
                assert math.isfinite(loss) and count>0
                total_loss+=loss*count;targets+=count
                details.append({'task':example['task'],'loss':loss,'target_tokens':count})
                del batch,output
    finally:model.train(was_training)
    return {'token_weighted_loss':total_loss/targets,'examples':len(examples),'target_tokens':targets,'details':details}


def setup_model(resume=None):
    set_seed(SEED)
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,trust_remote_code=False)
    if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
    quantization=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
    base=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=REVISION,quantization_config=quantization,
            device_map={'':0},dtype=torch.float16,attn_implementation='sdpa',trust_remote_code=False)
    base.config.use_cache=False
    base=prepare_model_for_kbit_training(base,use_gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False})
    if resume:model=PeftModel.from_pretrained(base,str(Path(resume)/'adapter'),is_trainable=True)
    else:model=get_peft_model(base,LoraConfig(r=8,lora_alpha=16,lora_dropout=0.05,target_modules=['q_proj','v_proj'],bias='none',task_type='CAUSAL_LM'))
    model.print_trainable_parameters()
    return model,tokenizer


def initialize_run(model,manifest,train_count,checkpoint_root,resume=None):
    root=Path(checkpoint_root);root.mkdir(parents=True,exist_ok=True)
    optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=LEARNING_RATE,weight_decay=0.)
    scaler=torch.amp.GradScaler('cuda')
    fingerprint=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()
    if resume:
        assert (Path(resume)/'COMPLETE').exists(),'Incomplete checkpoint: choose a completed checkpoint.'
        # Load only this notebook's own checkpoint; pickle state is not for arbitrary uploads.
        state=torch.load(Path(resume)/'training_state.pt',map_location='cpu',weights_only=False)
        assert state['manifest_fingerprint']==fingerprint and state['accumulation']==ACCUMULATION
        optimizer.load_state_dict(state['optimizer']);scaler.load_state_dict(state['scaler'])
        run=state['run'];torch.set_rng_state(state['torch_rng']);torch.cuda.set_rng_state_all(state['cuda_rng'])
        assert run['next_position']%ACCUMULATION==0 or run['next_position']==train_count
        del state
    else:
        assert not list(root.glob('step-*')),'Existing run found: resume it or choose a new output folder.'
        order=list(range(train_count));random.Random(SEED).shuffle(order)
        run={'order':order,'next_position':0,'optimizer_steps':0,'log':[],'baseline':None}
    assert sorted(run['order'])==list(range(train_count))
    return {'run':run,'optimizer':optimizer,'scaler':scaler,'root':root,'fingerprint':fingerprint}


def save_checkpoint(model,tokenizer,state):
    run=state['run'];destination=state['root']/f'step-{run["optimizer_steps"]:05d}'
    assert not destination.exists(),'Checkpoint already exists; use the latest checkpoint or a new run folder.'
    folder=state['root']/f'.incomplete-{run["optimizer_steps"]:05d}-{uuid.uuid4().hex}'
    folder.mkdir(exist_ok=False)
    model.save_pretrained(folder/'adapter');tokenizer.save_pretrained(folder/'tokenizer')
    torch.save({'optimizer':state['optimizer'].state_dict(),'scaler':state['scaler'].state_dict(),'run':run,
                'manifest_fingerprint':state['fingerprint'],'accumulation':ACCUMULATION,
                'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all()},folder/'training_state.pt')
    (folder/'COMPLETE').write_text('Checkpoint files saved successfully.\n')
    folder.rename(destination)
    (state['root']/'latest_checkpoint.txt').write_text(str(destination))
    print('Checkpoint:',destination,flush=True)


def train_epoch(model,tokenizer,examples,state):
    run=state['run'];optimizer=state['optimizer'];scaler=state['scaler']
    model.train();optimizer.zero_grad(set_to_none=True)
    gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()
    start=time.perf_counter();start_position=run['next_position'];pending_loss=0.;pending_tasks=[]
    for position in range(start_position,len(examples)):
        group_start=(position//ACCUMULATION)*ACCUMULATION
        group_size=min(ACCUMULATION,len(examples)-group_start)
        example=examples[run['order'][position]];batch=batch_for(example)
        with torch.autocast('cuda',dtype=torch.float16):
            output=model(**batch,use_cache=False);loss=output.loss
        assert torch.isfinite(loss).item(), 'Nonfinite loss; stop and resume the last complete checkpoint.'
        value=float(loss.detach().float());scaler.scale(loss/group_size).backward()
        pending_loss+=value;pending_tasks.append(example['task'])
        del batch,output,loss
        if (position+1)%ACCUMULATION==0 or position+1==len(examples):
            scaler.unscale_(optimizer)
            norm=torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],1.,error_if_nonfinite=True)
            scaler.step(optimizer);scaler.update();optimizer.zero_grad(set_to_none=True)
            run['next_position']=position+1;run['optimizer_steps']+=1
            run['log'].append({'step':run['optimizer_steps'],'examples_seen':position+1,'mean_example_loss':pending_loss/group_size,
                               'gradient_norm':float(norm),'tasks':pending_tasks})
            pending_loss=0.;pending_tasks=[]
            if run['optimizer_steps']%10==0 or run['optimizer_steps']==1:
                elapsed=time.perf_counter()-start;done=position+1-start_position
                remaining=elapsed/done*(len(examples)-position-1)/60
                print(f'Update {run["optimizer_steps"]}/1250 | examples {position+1}/10000 | loss {run["log"][-1]["mean_example_loss"]:.4f} | remaining ~{remaining:.1f} min',flush=True)
            run['peak_gpu_memory_gib']=max(run.get('peak_gpu_memory_gib',0.),torch.cuda.max_memory_allocated()/1024**3)
            if run['optimizer_steps']==1 or run['optimizer_steps']%50==0 or position+1==len(examples):save_checkpoint(model,tokenizer,state)
    assert run['next_position']==len(examples)
    run['peak_gpu_memory_gib']=max(run.get('peak_gpu_memory_gib',0.),torch.cuda.max_memory_allocated()/1024**3)
    print('One epoch complete. Peak GPU GiB:',round(run['peak_gpu_memory_gib'],2))


In [ ]:
assert "model" not in globals(), "Use a fresh runtime; do not load a second model over an existing one."
manifest, rows = load_dataset(dataset_dir)
model, tokenizer = setup_model(RESUME_FROM)
train_examples = prepare(rows["train"], tokenizer)
validation_examples = prepare(rows["validation"], tokenizer)
comparison_validation = sorted(validation_examples,
    key=lambda x: hashlib.sha256(x["task"].encode()).hexdigest())[:100]
state = initialize_run(model, manifest, len(train_examples), RUN_ROOT, RESUME_FROM)
print("Prepared:", len(train_examples), "training;", len(validation_examples), "development validation.")
print("Resuming at example:", state["run"]["next_position"])
print("No truncation. Prompt tokens are masked from loss.")


## Baseline on a fixed 100-example development subset
The exact same 100 examples are evaluated after training. The remaining validation
records remain available for final development evaluation; this is not a benchmark test set.


In [ ]:
if state["run"]["baseline"] is None:
    assert state["run"]["next_position"] == 0
    with model.disable_adapter():
        state["run"]["baseline"] = evaluate(model, comparison_validation)
    (RUN_ROOT / "baseline.json").write_text(json.dumps(state["run"]["baseline"], indent=2))
print("Baseline loss:", state["run"]["baseline"]["token_weighted_loss"])


## Train one epoch
Run this cell once. Expect a sustained run; estimates appear after the first updates.
Keep Colab connected. If interrupted or an error occurs, start a fresh runtime and resume
the latest COMPLETE checkpoint. Do not rerun this cell on a partially updated model.


In [ ]:
assert "training_cell_started" not in globals(), "Training was already started in this runtime. Resume from Drive in a fresh runtime."
training_cell_started = True
train_epoch(model, tokenizer, train_examples, state)


In [ ]:
assert state["run"]["next_position"] == 10000
after = evaluate(model, comparison_validation)
baseline = state["run"]["baseline"]
results = {
    "model": MODEL_ID, "revision": REVISION,
    "training_examples": 10000, "validation_available": 1000,
    "comparison_validation_examples": 100,
    "baseline": baseline, "after": after,
    "loss_change": after["token_weighted_loss"] - baseline["token_weighted_loss"],
    "optimizer_steps": state["run"]["optimizer_steps"],
    "training_peak_gpu_gib": state["run"]["peak_gpu_memory_gib"],
    "dataset_status": manifest["status"], "software": software,
    "meaning": "Development imitation loss only; no generated patches executed."
}
(RUN_ROOT / "results.json").write_text(json.dumps(results, indent=2))
final_dir = RUN_ROOT / "final_export"
final_dir.mkdir(exist_ok=False)
model.save_pretrained(final_dir / "adapter")
tokenizer.save_pretrained(final_dir / "tokenizer")
(final_dir / "results.json").write_text(json.dumps(results, indent=2))
(final_dir / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2))
print("Before:", baseline["token_weighted_loss"])
print("After:", after["token_weighted_loss"])
print("Change:", results["loss_change"])
print("Saved:", final_dir)


## Reload check and download
This checks the saved adapter on the same base weights. It does not independently
reload the entire quantized base model or demonstrate successful software repairs.


In [ ]:
import math, shutil
original_adapter = model.active_adapter
model.load_adapter(str(final_dir / "adapter"), adapter_name="reload_check", is_trainable=False)
try:
    model.set_adapter("reload_check")
    reloaded = evaluate(model, comparison_validation)
    difference = abs(reloaded["token_weighted_loss"] - after["token_weighted_loss"])
    assert math.isclose(reloaded["token_weighted_loss"], after["token_weighted_loss"], rel_tol=1e-5, abs_tol=1e-6)
finally:
    model.set_adapter(original_adapter)
(final_dir / "reload_check.json").write_text(json.dumps({"difference": difference, "match": True}, indent=2))
print("Saved-adapter reload: MATCH; difference:", difference)
archive = shutil.make_archive("/content/svaga-localized-qwen15b-10k-final", "zip", final_dir)
files.download(archive)


## Optional: evaluate all 1,000 development records
Run after the matched 100-example comparison and save. The full-set loss has no
full-set baseline in this notebook, so do not compare it directly to the 100-example baseline.
Actual repair quality still requires generating patches and executing tests in isolated tasks.


In [ ]:
full_validation = evaluate(model, validation_examples)
(RUN_ROOT / "full_development_validation.json").write_text(json.dumps(full_validation, indent=2))
print("Full development loss:", full_validation["token_weighted_loss"])
print("Examples:", full_validation["examples"])
